# Step 4: Understand usage and calculate electricity costs

**All usage and plans in this lesson are fictional.** They are not offers from the
providers in your PDFs and are not estimates of your household's bills.

Select `.venv` and run cells in order with **Shift+Enter**. This notebook is
self-contained and makes no API calls. You do not need to run 01, 02, or 03 first.
We will make sample data, validate it, calculate a bill, then compare a full year.

## 1. Import the building blocks
`Decimal` keeps decimal arithmetic predictable. Enter rates as strings, such as
`Decimal("0.12")`, instead of converting a binary floating-point number.
`usage.py` checks the input; `pricing.py` applies explicit arithmetic rules.

In [1]:
from pathlib import Path
from decimal import Decimal
import csv
import json
from html import escape
from IPython.display import display, HTML
from electricity_optimizer.usage import MonthlyUsage, UsageYear, load_usage_csv
from electricity_optimizer.pricing import DemoPlan, calculate_month, compare_plans

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "electricity_optimizer").is_dir():
    raise RuntimeError("Open this notebook from the project root.")
output_dir = PROJECT_ROOT / "output" / "lesson04_synthetic"
output_dir.mkdir(parents=True, exist_ok=True)

# Render small readable tables without adding a dataframe dependency.
def show_table(headers, rows):
    def row(values, tag):
        return "<tr>" + "".join(f"<{tag} style='padding:6px 12px;text-align:left'>{escape(str(v))}</{tag}>" for v in values) + "</tr>"
    display(HTML("<table>" + row(headers, "th") + "".join(row(r, "td") for r in rows) + "</table>"))

## 2. Create an example year
These invented values illustrate higher summer usage, not a statistical household
profile. The notebook creates a CSV intermediate so we can learn the import flow.
Change values here to experiment. Rerunning replaces only this lesson's sample CSV.
`kWh` measures energy used; a price in dollars per kWh is a different quantity.

In [2]:
sample_kwh = [850, 720, 650, 700, 900, 1200, 1500, 1600, 1250, 950, 750, 900]
usage_path = output_dir / "SYNTHETIC_usage_2025.csv"
with usage_path.open("w", encoding="utf-8", newline="") as stream:
    writer = csv.writer(stream)
    writer.writerow(["month", "kwh"])
    writer.writerows((f"2025-{month:02d}", kwh) for month, kwh in enumerate(sample_kwh, start=1))
print("Created synthetic data:", usage_path)
print(usage_path.read_text(encoding="utf-8"))

Created synthetic data: c:\Users\Nyalo\VSCode_Projects\Electricity_Agreement_Optimizer\output\lesson04_synthetic\SYNTHETIC_usage_2025.csv
month,kwh
2025-01,850
2025-02,720
2025-03,650
2025-04,700
2025-05,900
2025-06,1200
2025-07,1500
2025-08,1600
2025-09,1250
2025-10,950
2025-11,750
2025-12,900



## 3. Load and validate the CSV
We require exactly 12 consecutive months, no duplicates, valid month labels, and
finite nonnegative kWh. The loader sorts rows by month. Zero usage is allowed;
missing usage is not silently filled with zero.

In [3]:
usage = load_usage_csv(usage_path)
show_table(["Month", "Synthetic usage (kWh)"], [(m.month, m.kwh) for m in usage.months])
print("Annual kWh:", sum(m.kwh for m in usage.months))

Month,Synthetic usage (kWh)
2025-01,850
2025-02,720
2025-03,650
2025-04,700
2025-05,900
2025-06,1200
2025-07,1500
2025-08,1600
2025-09,1250
2025-10,950


Annual kWh: 11970


## 4. Define two fictional plans
Both use USD and the same fictional market. Rates remain constant for all 12 months.
Delivery charges are explicitly included. Taxes, switching fees, deposits, variable
rates, time-of-use rates, tiers and renewal changes are outside this first lesson.

**Simple:** lower energy rate, no credit.
**Threshold:** higher energy rate, with a $50 credit in months using at least 1,000 kWh.
These values are invented; we have not converted your PDF terms into pricing rules.

Rounding assumption: round each charge component to cents, half up. Apply the
credit afterward, capped at the month's subtotal so it cannot create a negative bill.

In [4]:
simple = DemoPlan(name="Fictional Simple", energy_usd_per_kwh="0.12",
    base_usd_per_month="10", delivery_usd_per_kwh="0.05", delivery_usd_per_month="5",
    credit_usd="0", credit_min_kwh=None)
threshold = DemoPlan(name="Fictional Threshold", energy_usd_per_kwh="0.15",
    base_usd_per_month="5", delivery_usd_per_kwh="0.05", delivery_usd_per_month="5",
    credit_usd="50", credit_min_kwh="1000")
plans = [simple, threshold]
show_table(["Fictional plan", "Energy $/kWh", "Base $/month", "Delivery $/kWh", "Delivery $/month", "Credit $", "Minimum kWh"],
    [(p.name, p.energy_usd_per_kwh, p.base_usd_per_month, p.delivery_usd_per_kwh,
      p.delivery_usd_per_month, p.credit_usd, p.credit_min_kwh) for p in plans])

Fictional plan,Energy $/kWh,Base $/month,Delivery $/kWh,Delivery $/month,Credit $,Minimum kWh
Fictional Simple,0.12,10,0.05,5,0,None
Fictional Threshold,0.15,5,0.05,5,50,1000


## 5. Follow one bill by hand
For the threshold plan at **1,200 kWh**:
- Energy: 1,200 ? $0.15 = $180.
- Base fee: $5.
- Delivery: 1,200 ? $0.05 + $5 = $65.
- Credit: $50, because 1,200 is at least 1,000.
- Total: $180 + $5 + $65 ? $50 = **$200**.

Change the usage below and compare the line items. Python computes this bill; an
AI model does not decide the arithmetic.

In [5]:
example_month = MonthlyUsage(month="2025-06", kwh="1200")
example_bill = calculate_month(example_month, threshold)
show_table(["Component", "USD"], [
    ("Energy", example_bill.energy_usd), ("Base", example_bill.base_usd),
    ("Delivery", example_bill.delivery_usd), ("Credit (subtract)", example_bill.credit_usd),
    ("Total", example_bill.total_usd)])

Component,USD
Energy,180.00
Base,5.00
Delivery,65.00
Credit (subtract),50.00
Total,200.00


## 6. Inspect the credit boundary
Predict the bills at 999, 1,000 and 1,001 kWh before running the cell. The threshold
is inclusive: exactly 1,000 qualifies. A lower bill at the threshold is a consequence
of this fictional rule, not advice to consume more electricity.

In [6]:
boundary_rows = []
for kwh in ["999", "1000", "1001"]:
    month = MonthlyUsage(month="2025-01", kwh=kwh)
    a, b = calculate_month(month, simple), calculate_month(month, threshold)
    boundary_rows.append((kwh, a.total_usd, b.credit_usd, b.total_usd))
show_table(["kWh", "Simple total $", "Threshold credit $", "Threshold total $"], boundary_rows)

kWh,Simple total $,Threshold credit $,Threshold total $
999,184.83,0.00,209.80
1000,185.00,50.00,160.00
1001,185.17,50.00,160.20


## 7. Compare all 12 months
Calculate credits separately for each month. Annual-average usage cannot tell us
which months qualify. Rank by the sum of monthly bills under the stated assumptions;
a tie in annual cost is a tie, even though output order uses names for consistency.

In [7]:
results = compare_plans(usage, plans)
by_name = {r.plan.name: r for r in results}
show_table(["Month", "kWh", "Simple $", "Threshold $", "Threshold credit $"],
    [(a.month, a.kwh, a.total_usd, b.total_usd, b.credit_usd)
     for a, b in zip(by_name[simple.name].bills, by_name[threshold.name].bills)])
show_table(["Fictional plan", "Annual total USD", "Months credited"],
    [(r.plan.name, r.annual_usd, sum(b.credit_usd > 0 for b in r.bills)) for r in results])
print("Annual difference between these two fictional plans: $", results[-1].annual_usd - results[0].annual_usd)

Month,kWh,Simple $,Threshold $,Threshold credit $
2025-01,850,159.50,180.00,0.00
2025-02,720,137.40,154.00,0.00
2025-03,650,125.50,140.00,0.00
2025-04,700,134.00,150.00,0.00
2025-05,900,168.00,190.00,0.00
2025-06,1200,219.00,200.00,50.00
2025-07,1500,270.00,260.00,50.00
2025-08,1600,287.00,280.00,50.00
2025-09,1250,227.50,210.00,50.00
2025-10,950,176.50,200.00,0.00


Fictional plan,Annual total USD,Months credited
Fictional Simple,2214.90,0
Fictional Threshold,2314.00,4


Annual difference between these two fictional plans: $ 99.10


## 8. Experiment: what if consumption changes?
Change the multiplier, then rerun this cell. This is a uniform teaching scenario,
not a forecast. The original sample data remains unchanged. Observe whether a plan's
ranking changes as more months cross its credit threshold.

In [8]:
USAGE_MULTIPLIER = Decimal("1.20")
scenario = UsageYear(months=tuple(MonthlyUsage(month=m.month, kwh=m.kwh * USAGE_MULTIPLIER) for m in usage.months))
scenario_results = compare_plans(scenario, plans)
show_table(["Fictional plan", "Scenario annual USD", "Months credited"],
    [(r.plan.name, r.annual_usd, sum(b.credit_usd > 0 for b in r.bills)) for r in scenario_results])

Fictional plan,Scenario annual USD,Months credited
Fictional Threshold,2592.80,8
Fictional Simple,2621.88,0


## 9. Save the base comparison
Save assumptions, synthetic usage, plan definitions and monthly line items together.
This report uses the original sample usage, not the multiplier scenario. Running
again replaces the lesson report. Decimal values are saved as strings to retain precision.

In [9]:
report = {
    "synthetic": True,
    "description": "Teaching example only; not real provider offers or household estimates.",
    "assumptions": ["USD; same fictional market; rates constant for 12 months",
        "Energy, base and delivery included; taxes and switching fees excluded",
        "Each component rounded to cents half up; credit capped at monthly subtotal"],
    "usage": usage.model_dump(mode="json"),
    "comparisons": [r.model_dump(mode="json") for r in results],
}
report_path = output_dir / "SYNTHETIC_comparison.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
assert json.loads(report_path.read_text(encoding="utf-8")) == report
print("Saved:", report_path)

Saved: c:\Users\Nyalo\VSCode_Projects\Electricity_Agreement_Optimizer\output\lesson04_synthetic\SYNTHETIC_comparison.json


## Pause here and explain what happened
1. Why does the threshold plan's bill drop at 1,000 kWh?
2. Is the plan with the lower energy rate always cheaper for a particular month?
3. What changed when you multiplied consumption by 1.20?

We now have a deterministic calculation foundation. Before applying it to actual
agreements, we need complete reviewed pricing terms, compatible service territories,
and applicable fees. The next extension will connect reviewed inputs to the workflow;
this notebook does not automatically approve or price any of your PDFs.